# Wildfire Ignition Prediction — Exploratory Data Analysis

**Project**: California Wildfire Ignition Probability (H3 Hex Grid, 7-day horizon)

This notebook walks through the full EDA pipeline in six sections:

1. **Grid Exploration** — Visualise the H3 hex grid, cell counts, and burnable area
2. **Fire Data EDA** — Temporal patterns, ignition locations, seasonal distribution
3. **Weather EDA** — Feature distributions, temporal trends, spatial variation
4. **Label Distribution** — Positive rate, spatio-temporal clustering of ignitions
5. **Feature Correlations** — Heatmap, top features, multicollinearity check
6. **Key Findings** — Summary for model design decisions

---
> **Before running**: ensure the virtual environment is active and interim/processed
> data has been generated by the pipeline (`scripts/run_pipeline.py`).

In [ ]:
# Standard imports
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Project root on path
PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

# Plot style
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.2)
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 120

INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

print("Project root:", PROJECT_ROOT)
print("Interim dir exists:", INTERIM_DIR.exists())

---
## Section 1: Grid Exploration

The prediction unit is an **H3 resolution-6 hexagonal cell** (~4 km edge length, ~36 km² area).
California has roughly 30k–40k burnable cells after excluding water and dense-urban land cover.

Key questions:
- How many cells are in the California bounding box?
- How many are excluded as non-burnable?
- What is the geographic distribution?

In [ ]:
# Load the H3 grid (generated by preprocessing stage)
grid_path = INTERIM_DIR / "ca_grid.parquet"

if grid_path.exists():
    grid_df = pd.read_parquet(grid_path)
    print(f"Grid loaded: {len(grid_df):,} burnable cells")
    print(grid_df.head())
else:
    print("Grid file not found — generating a synthetic example for illustration.")
    # Synthetic demo: random lat/lon within California bounding box
    rng = np.random.default_rng(42)
    n = 5000
    grid_df = pd.DataFrame({
        "cell_id": [f"cell_{i}" for i in range(n)],
        "lat": rng.uniform(32.5, 42.0, n),
        "lon": rng.uniform(-124.5, -114.1, n),
    })
    print(f"Synthetic grid: {len(grid_df):,} cells")

In [ ]:
# Geographic distribution of cells
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].scatter(grid_df["lon"], grid_df["lat"], s=0.3, alpha=0.4, c="steelblue")
axes[0].set_xlabel("Longitude")
axes[0].set_ylabel("Latitude")
axes[0].set_title(f"H3 Burnable Cells (n={len(grid_df):,})")

# Latitude distribution
axes[1].hist(grid_df["lat"], bins=50, color="steelblue", edgecolor="white")
axes[1].set_xlabel("Latitude")
axes[1].set_ylabel("Cell Count")
axes[1].set_title("Latitude Distribution of Burnable Cells")

plt.tight_layout()
plt.show()

print(f"\nBounding box:")
print(f"  Lat: [{grid_df['lat'].min():.2f}, {grid_df['lat'].max():.2f}]")
print(f"  Lon: [{grid_df['lon'].min():.2f}, {grid_df['lon'].max():.2f}]")

---
## Section 2: Fire Data EDA

Fire ignition data comes from **MODIS MOD14A1** (Terra Thermal Anomalies, 1 km daily).
We use fire pixels as proxies for ignition locations.

Key questions:
- What is the seasonality of ignitions? (expected peak: July–October)
- How are ignitions distributed spatially?
- Are there notable year-over-year trends (2015–2023)?

In [ ]:
fire_path = INTERIM_DIR / "fire_ignitions.parquet"

if fire_path.exists():
    fire_df = pd.read_parquet(fire_path)
    fire_df["fire_date"] = pd.to_datetime(fire_df["fire_date"])
    print(f"Fire records: {len(fire_df):,}")
    print(f"Date range: {fire_df['fire_date'].min()} → {fire_df['fire_date'].max()}")
    print(fire_df.head())
else:
    print("Fire data not found — generating synthetic data for illustration.")
    rng = np.random.default_rng(42)
    n_fires = 2000
    # California fires cluster in summer/fall
    months = rng.choice(
        range(1, 13), n_fires,
        p=[0.01, 0.01, 0.02, 0.03, 0.05, 0.07, 0.12, 0.18, 0.22, 0.15, 0.09, 0.05]
    )
    years = rng.integers(2015, 2024, n_fires)
    days = rng.integers(1, 28, n_fires)
    fire_df = pd.DataFrame({
        "cell_id": [f"cell_{rng.integers(0, 5000)}" for _ in range(n_fires)],
        "fire_date": pd.to_datetime({
            "year": years, "month": months, "day": days
        }),
        "lat": rng.uniform(32.5, 42.0, n_fires),
        "lon": rng.uniform(-124.5, -114.1, n_fires),
    })
    print(f"Synthetic fire data: {len(fire_df):,} records")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Monthly seasonality
monthly = fire_df.groupby(fire_df["fire_date"].dt.month).size()
monthly.index = ["Jan","Feb","Mar","Apr","May","Jun",
                 "Jul","Aug","Sep","Oct","Nov","Dec"]
monthly.plot(kind="bar", ax=axes[0], color="tomato", edgecolor="white")
axes[0].set_title("Ignitions by Month (all years)")
axes[0].set_xlabel("Month")
axes[0].set_ylabel("# Ignitions")
axes[0].tick_params(axis="x", rotation=45)

# Annual trend
annual = fire_df.groupby(fire_df["fire_date"].dt.year).size()
annual.plot(kind="bar", ax=axes[1], color="darkorange", edgecolor="white")
axes[1].set_title("Ignitions by Year")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("# Ignitions")
axes[1].tick_params(axis="x", rotation=45)

# Spatial distribution
if "lat" in fire_df.columns and "lon" in fire_df.columns:
    axes[2].scatter(fire_df["lon"], fire_df["lat"], s=1, alpha=0.3, c="tomato")
    axes[2].set_xlabel("Longitude")
    axes[2].set_ylabel("Latitude")
    axes[2].set_title("Ignition Locations (all years)")

plt.tight_layout()
plt.show()

**Observations**:
- Strong summer–fall seasonality (peak August–October) consistent with California fire climate.
- Year-over-year variation reflects drought cycles and fuel accumulation.
- Spatial clustering in the Sierra Nevada foothills, Coast Ranges, and Southern California.

---
## Section 3: Weather EDA

Weather features come from **NOAA CFSv2** 7-day forecasts.  We examine:
- Distribution of temperature, humidity, wind, and precipitation
- Correlation with fire days vs. non-fire days
- Temporal trends (drought years visible in precipitation)

In [ ]:
weather_path = INTERIM_DIR / "weather_aligned.parquet"

if weather_path.exists():
    weather_df = pd.read_parquet(weather_path)
    weather_df["date"] = pd.to_datetime(weather_df["date"])
    print(f"Weather rows: {len(weather_df):,}")
    print(weather_df.describe())
else:
    print("Weather data not found — generating synthetic data.")
    rng = np.random.default_rng(42)
    n_cells = 50
    dates = pd.date_range("2015-01-01", "2023-12-31", freq="D")
    rows = []
    for cell_i in range(n_cells):
        doy = np.arange(1, len(dates) + 1) % 365
        # Temperature peaks in summer (peak DOY ~200)
        tmp = 290 + 15 * np.sin(2 * np.pi * (doy - 80) / 365) + rng.normal(0, 3, len(dates))
        rh = 60 - 20 * np.sin(2 * np.pi * (doy - 80) / 365) + rng.normal(0, 8, len(dates))
        wnd = rng.exponential(3, len(dates))
        apcp = rng.exponential(1, len(dates))
        for i, d in enumerate(dates):
            rows.append({
                "cell_id": f"cell_{cell_i}",
                "date": d,
                "tmp2m": tmp[i],
                "rh2m": np.clip(rh[i], 5, 100),
                "wnd10m": wnd[i],
                "apcp": apcp[i],
            })
    weather_df = pd.DataFrame(rows)
    print(f"Synthetic weather: {len(weather_df):,} rows")

In [ ]:
# Distribution plots for weather variables
weather_cols = [c for c in ["tmp2m", "rh2m", "wnd10m", "apcp"] if c in weather_df.columns]
labels_map = {
    "tmp2m": "Temperature (K)",
    "rh2m": "Relative Humidity (%)",
    "wnd10m": "Wind Speed (m/s)",
    "apcp": "Precipitation (mm)",
}

fig, axes = plt.subplots(1, len(weather_cols), figsize=(16, 4))
if len(weather_cols) == 1:
    axes = [axes]

# Sample for speed
sample = weather_df.sample(min(50000, len(weather_df)), random_state=42)

for ax, col in zip(axes, weather_cols):
    ax.hist(sample[col].dropna(), bins=60, color="steelblue", edgecolor="white", density=True)
    ax.set_xlabel(labels_map.get(col, col))
    ax.set_ylabel("Density")
    ax.set_title(f"{labels_map.get(col, col)} Distribution")

plt.tight_layout()
plt.show()

print("\nWeather summary statistics (sample):")
print(sample[weather_cols].describe().round(2))

In [ ]:
# Seasonal patterns: average weather by month across all cells
weather_df["month"] = weather_df["date"].dt.month
monthly_weather = weather_df.groupby("month")[weather_cols].mean()
monthly_weather.index = ["Jan","Feb","Mar","Apr","May","Jun",
                          "Jul","Aug","Sep","Oct","Nov","Dec"]

fig, axes = plt.subplots(1, len(weather_cols), figsize=(16, 4))
if len(weather_cols) == 1:
    axes = [axes]

colors = ["tomato", "steelblue", "mediumseagreen", "orchid"]
for ax, col, color in zip(axes, weather_cols, colors):
    monthly_weather[col].plot(ax=ax, marker="o", color=color, linewidth=2)
    ax.set_xlabel("Month")
    ax.set_ylabel(labels_map.get(col, col))
    ax.set_title(f"Monthly Mean {labels_map.get(col, col)}")
    ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

**Observations**:
- Temperature peaks in July–August, humidity is lowest in summer → dry, hot conditions align with fire season.
- Precipitation is concentrated in winter months; near-zero in summer (Mediterranean climate).
- Wind speed is more uniform seasonally but spikes drive extreme fire behavior (Diablo/Santa Ana winds).

---
## Section 4: Label Distribution

The target is highly imbalanced: label=1 when a cell has an ignition in the next 7 days.
Expected positive rate: **< 1%** of (cell, date) pairs.

Key questions:
- What is the actual positive rate?
- Is the imbalance uniform across months / years?
- Are positive labels spatially clustered?

In [ ]:
labels_path = PROCESSED_DIR / "labels_train.parquet"

if labels_path.exists():
    labels_df = pd.read_parquet(labels_path)
    labels_df["date"] = pd.to_datetime(labels_df["date"])
    print(f"Label rows: {len(labels_df):,}")
    print(f"Positive rate: {labels_df['label'].mean()*100:.4f}%")
    print(f"Positive samples: {labels_df['label'].sum():,}")
else:
    print("Labels not found — generating synthetic labels.")
    n_cells = 100
    dates = pd.date_range("2015-01-01", "2021-12-31", freq="D")
    rng = np.random.default_rng(42)
    rows = []
    for cell_i in range(n_cells):
        # Label=1 with higher probability in summer
        for d in dates:
            summer_boost = 1.0 + 3.0 * np.sin(2 * np.pi * (d.dayofyear - 80) / 365)
            base_rate = 0.005
            prob = min(base_rate * max(summer_boost, 0.1), 1.0)
            rows.append({"cell_id": f"cell_{cell_i}", "date": d, "label": int(rng.random() < prob)})
    labels_df = pd.DataFrame(rows)
    print(f"Synthetic labels: {len(labels_df):,} rows, pos rate = {labels_df['label'].mean()*100:.4f}%")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Overall class balance
counts = labels_df["label"].value_counts()
axes[0].bar(["No Ignition (0)", "Ignition (1)"], counts.values,
             color=["steelblue", "tomato"], edgecolor="white")
axes[0].set_yscale("log")
axes[0].set_ylabel("Count (log scale)")
axes[0].set_title(f"Class Distribution (pos rate = {labels_df['label'].mean()*100:.3f}%)")

# Positive rate by month
labels_df["month"] = labels_df["date"].dt.month
monthly_pos = labels_df.groupby("month")["label"].mean() * 100
monthly_pos.index = ["Jan","Feb","Mar","Apr","May","Jun",
                      "Jul","Aug","Sep","Oct","Nov","Dec"]
monthly_pos.plot(kind="bar", ax=axes[1], color="tomato", edgecolor="white")
axes[1].set_title("Positive Rate (%) by Month")
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Positive Rate (%)")
axes[1].tick_params(axis="x", rotation=45)

# Positive rate by year
labels_df["year"] = labels_df["date"].dt.year
annual_pos = labels_df.groupby("year")["label"].mean() * 100
annual_pos.plot(kind="bar", ax=axes[2], color="darkorange", edgecolor="white")
axes[2].set_title("Positive Rate (%) by Year")
axes[2].set_xlabel("Year")
axes[2].set_ylabel("Positive Rate (%)")
axes[2].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

**Observations**:
- The positive rate is **< 1%** — severe class imbalance requiring `scale_pos_weight` in the model.
- Clear summer-fall peak (Jul–Oct), consistent with MODIS fire observations.
- Year-to-year variation reflects drought cycles (2020–2021 notably elevated).
- **Metric implication**: ROC-AUC is misleading at this imbalance ratio; PR-AUC is the primary metric.

---
## Section 5: Feature Correlations

Before training, we examine:
1. Correlation of each feature with the label
2. Feature-feature correlation (multicollinearity)
3. Distribution of features conditioned on label (fire vs. no-fire days)

In [ ]:
features_path = PROCESSED_DIR / "features_train.parquet"

if features_path.exists():
    feat_df = pd.read_parquet(features_path)
    print(f"Feature matrix: {feat_df.shape}")
    print(f"Columns: {list(feat_df.columns[:20])} ...")
else:
    print("Features not found — generating synthetic feature matrix.")
    rng = np.random.default_rng(42)
    n = 10000
    labels = rng.binomial(1, 0.008, n)
    feat_df = pd.DataFrame({
        "cell_id": [f"cell_{rng.integers(0, 100)}" for _ in range(n)],
        "date": pd.date_range("2015-01-01", periods=n, freq="D")[:n],
        "tmp2m_max_7d": 295 + 10 * labels + rng.normal(0, 5, n),
        "rh2m_min_7d": 40 - 15 * labels + rng.normal(0, 8, n),
        "wnd10m_max_7d": 5 + 3 * labels + rng.exponential(2, n),
        "apcp_sum_7d": rng.exponential(5, n),
        "tmp2m_roll7_mean": 292 + 8 * labels + rng.normal(0, 4, n),
        "rh2m_roll7_mean": 45 - 12 * labels + rng.normal(0, 7, n),
        "apcp_roll30_sum": rng.exponential(15, n),
        "wnd10m_roll7_mean": 3 + 2 * labels + rng.exponential(1, n),
        "elevation_m": rng.uniform(0, 3000, n),
        "slope_deg": rng.uniform(0, 45, n),
        "aspect_sin": rng.uniform(-1, 1, n),
        "aspect_cos": rng.uniform(-1, 1, n),
        "fuel_model": rng.integers(0, 10, n),
        "canopy_cover_pct": rng.uniform(0, 100, n),
        "neighbor_fire_count_7d": rng.integers(0, 3, n),
        "road_density_km_per_km2": rng.exponential(1, n),
        "pop_density_per_km2": rng.exponential(0.5, n),
        "doy_sin": np.sin(2 * np.pi * np.arange(n) / 365.25),
        "doy_cos": np.cos(2 * np.pi * np.arange(n) / 365.25),
        "month_sin": rng.uniform(-1, 1, n),
        "month_cos": rng.uniform(-1, 1, n),
        "label": labels,
    })
    print(f"Synthetic features: {feat_df.shape}")

In [ ]:
# Select numeric feature columns (exclude metadata)
meta_cols = {"cell_id", "date", "label"}
feat_cols = [c for c in feat_df.columns if c not in meta_cols and feat_df[c].dtype in ["float64", "int64", "float32", "int32"]]

# Point-biserial correlations with label
correlations = feat_df[feat_cols + ["label"]].corr()["label"].drop("label").sort_values()

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Top / bottom correlations
n_show = min(15, len(correlations))
top_bottom = pd.concat([correlations.head(n_show // 2), correlations.tail(n_show // 2)])
colors = ["tomato" if v > 0 else "steelblue" for v in top_bottom.values]
top_bottom.plot(kind="barh", ax=axes[0], color=colors, edgecolor="white")
axes[0].axvline(0, color="black", linewidth=0.8)
axes[0].set_title("Feature Correlation with Label (top/bottom)")
axes[0].set_xlabel("Pearson Correlation")

# Feature-feature correlation heatmap (select key features)
heatmap_cols = feat_cols[:min(12, len(feat_cols))]
corr_matrix = feat_df[heatmap_cols].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, ax=axes[1],
    cmap="RdBu_r", center=0, vmin=-1, vmax=1,
    annot=True if len(heatmap_cols) <= 10 else False,
    fmt=".2f", linewidths=0.5, square=True,
)
axes[1].set_title("Feature-Feature Correlation (lower triangle)")
axes[1].tick_params(axis="x", rotation=45)
axes[1].tick_params(axis="y", rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Distribution of top features conditioned on label
top_features = correlations.abs().sort_values(ascending=False).head(4).index.tolist()
label_display = {0: "No Ignition", 1: "Ignition"}

fig, axes = plt.subplots(1, len(top_features), figsize=(16, 4))
if len(top_features) == 1:
    axes = [axes]

# Sample for speed
sample = feat_df.sample(min(30000, len(feat_df)), random_state=42)

for ax, col in zip(axes, top_features):
    for label_val, color in [(0, "steelblue"), (1, "tomato")]:
        subset = sample[sample["label"] == label_val][col].dropna()
        ax.hist(subset, bins=40, alpha=0.6, density=True,
                label=label_display[label_val], color=color, edgecolor="white")
    ax.set_xlabel(col)
    ax.set_ylabel("Density")
    ax.set_title(f"{col}")
    ax.legend(fontsize=9)

plt.suptitle("Top Features: Distribution by Class", y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

---
## Section 6: Key Findings & Design Decisions

This section summarises the EDA insights that directly inform model design.

In [ ]:
# Summary statistics table
summary = {
    "Metric": [
        "Burnable H3 cells",
        "Date range",
        "Total (cell, date) pairs",
        "Positive rate",
        "Scale_pos_weight (approx)",
        "Fire season months",
        "Top correlated feature",
    ],
    "Value": [
        f"{len(grid_df):,}",
        f"{fire_df['fire_date'].min().date()} → {fire_df['fire_date'].max().date()}",
        f"{len(labels_df):,}",
        f"{labels_df['label'].mean()*100:.4f}%",
        f"{(1 - labels_df['label'].mean()) / labels_df['label'].mean():.0f}",
        "July – October",
        correlations.abs().idxmax() if len(correlations) > 0 else "N/A",
    ]
}
pd.DataFrame(summary).style.set_caption("EDA Summary").hide(axis="index")

### Key Design Decisions Informed by EDA

| Finding | Design Decision |
|---------|----------------|
| < 1% positive rate | Use `scale_pos_weight = neg/pos`; optimise PR-AUC not accuracy |
| Strong seasonality | Include sin/cos DOY and month encodings as features |
| High tmp2m / low rh2m correlation with ignitions | Include both forecast AND lagged rolling weather features |
| Spatial clustering | Include H3 neighbor fire count as spatial feature |
| Year-to-year variation | Time-based train/val/test split (2015–2021 / 2022 / 2023) |
| Precipitation near-zero in fire season | Rolling 30-day precipitation sum captures drought conditions |

### Next Steps

1. Run `scripts/run_pipeline.py --skip-download` (after downloading data)
2. Train baseline logistic regression and LightGBM
3. Evaluate PR-AUC on 2022 validation set
4. Tune hyperparameters (depth, learning rate, subsampling)
5. Final evaluation on held-out 2023 test set